In [102]:
import pandas as pd
import numpy as np


In [103]:
recipes = pd.read_csv('scraped_recipe_ingredients.csv')


In [104]:
recipes['tags'] = recipes['tags'].apply(eval)
all_tags = recipes['tags'].explode().unique()
tags = pd.DataFrame(all_tags, columns=['tag'])
tags['t_id'] = tags.index+1
tags = tags[['t_id', 'tag']]
tags.to_csv('tags.csv', index=False)

In [105]:
import inflect
from collections import defaultdict

# Get unique ingredients
recipes['ingredients'] = recipes['ingredients'].apply(eval)
recipes['amount_list'] = recipes['amount_list'].apply(eval)
recipes['unit_list'] = recipes['unit_list'].apply(eval)

ingredients = recipes['ingredients'].explode().unique()

# Get singular forms
p = inflect.engine()

def to_singular(word):
    word = word.strip().lower()
    singular = p.singular_noun(word)
    return singular if singular else word

def pick_canonical(variants):
    """Prefer the already-singular form; fall back to shortest."""
    for v in variants:
        if to_singular(v.lower()) == v.lower():
            return v
    return min(variants, key=len)


# Remove duplicates by mapping to singular forms, then pick a canonical variant for each group
singular_map = defaultdict(list)
for ing in ingredients:
    key = to_singular(ing.lower())
    singular_map[key].append(ing)

cleaned = [pick_canonical(variants) for variants in singular_map.values()]
ing_df = pd.DataFrame(cleaned, columns=['ingredient'])
ing_df['i_id'] = ing_df.index+1


ing_df = ing_df[['i_id', 'ingredient']]
ing_df.to_csv("ingredients_cleaned.csv", index=False)


In [106]:
# Map recipes to ingredient IDs
recipe_ingredients = []
for recipe in recipes.itertuples():
    r_id = recipe.id
    ingredient_list = recipe.ingredients
    amount_list = recipe.amount_list
    unit_list = recipe.unit_list
    for i in range(len(ingredient_list)):
        print(f'{recipe.name}, {i}')
        ing = ingredient_list[i].lower().strip()
        amt = amount_list[i].lower().strip()
        unit = unit_list[i].lower().strip()
        ing_id = ing_df[ing_df['ingredient'] == pick_canonical(singular_map[to_singular(ing.lower())])]['i_id'].values[0]
        recipe_ingredients.append({'r_id': r_id, 'i_id': ing_id, 'amount': amt, 'unit': unit})
        

recipe_ingredients_df = pd.DataFrame(recipe_ingredients)
recipe_ingredients_df.to_csv('recipe_ingredients.csv', index=False)



easy   delicious orange slice cookies, 0
easy   delicious orange slice cookies, 1
easy   delicious orange slice cookies, 2
easy   delicious orange slice cookies, 3
easy   delicious orange slice cookies, 4
tomato salad with chocolate dressing, 0
tomato salad with chocolate dressing, 1
tomato salad with chocolate dressing, 2
tomato salad with chocolate dressing, 3
tomato salad with chocolate dressing, 4
tomato salad with chocolate dressing, 5
tomato salad with chocolate dressing, 6
tomato salad with chocolate dressing, 7
tomato salad with chocolate dressing, 8
tomato salad with chocolate dressing, 9
oat bran hot cereal, 0
oat bran hot cereal, 1
oat bran hot cereal, 2
a homemade   sundae, 0
a homemade   sundae, 1
a homemade   sundae, 2
a homemade   sundae, 3
a homemade   sundae, 4
a homemade   sundae, 5
broccoli and green bean polka, 0
broccoli and green bean polka, 1
broccoli and green bean polka, 2
broccoli and green bean polka, 3
broccoli and green bean polka, 4
broccoli and green bean

In [107]:
# Map recipes to tag IDs
recipe_tags = []
for recipe in recipes.itertuples():
    r_id = recipe.id
    for tag in recipe.tags:
        t_id = tags[tags['tag'] == tag]['t_id'].values[0]
        recipe_tags.append({'r_id': r_id, 't_id': t_id})

recipe_tags_df = pd.DataFrame(recipe_tags)
recipe_tags_df.to_csv('recipe_tags.csv', index=False)



In [108]:
recipes.rename(columns={'id': 'r_id', 'name': 'r_name', 'minutes': 'cooking_time', 'steps': 'instructions'}, inplace=True)
recipes['writer_id'] = 1
recipes[['r_id', 'r_name', 'instructions', 'description', 'cooking_time', 'writer_id']].to_csv('recipes.csv', index=False)


In [109]:
user_df = pd.DataFrame({'u_id': [1,2,3], 'username': ['konradfhp', 'esintao', 'dualipa'], 'email': ['bjv627@alumni.ku.dk', 'cxq772@alumni.ku.dk', 'dualipa@example.com'], 'password': ['berry','banana', 'password123']})

household_df = pd.DataFrame({'h_id': [1,2], 'h_name': ['dis2026', 'thelipahouse'], 'h_password': ['bananaberry', 'password123']})

member_of_df = pd.DataFrame({'u_id': [1,2,3], 'h_id': [1,1,2]})

review_df = pd.DataFrame({'r_id': [8401, 245863], 'u_id': [3, 2], 'stars': [5, 4], 'comment': ['Easy to make. Highly recommend!', 'Nice, though a bit difficult to make.']})

stock_df = pd.DataFrame({'s_id': [1,2,3,4,5], 'h_id': [1,1,1,2,2], 'i_id': [265,249,1714,1605,1385], 'quantity':[5,1,50,200,2], 'unit': ['whole','kg','g','g','l']})


user_df.to_csv('users.csv', index=False)
household_df.to_csv('households.csv', index=False)
member_of_df.to_csv('member_of.csv', index=False)
review_df.to_csv('reviews.csv', index=False)
stock_df.to_csv('stock.csv', index=False)